# EDA e preparação da Bank Marketing

A base pública é usada somente como proxy histórico de resposta a campanha. O target `y` não representa aprovação, proposta ou contratação de crédito.

In [ ]:
import json
import os
import sys
from collections import Counter
from pathlib import Path

working_directory = Path.cwd().resolve()
repo_root = next(
    candidate
    for candidate in (working_directory, *working_directory.parents)
    if (candidate / 'pyproject.toml').is_file()
)
sys.path.insert(0, str(repo_root / 'src'))
from responsible_next_step.bank_marketing import prepare_bank_marketing

csv_path = Path(os.environ.get(
    'BANK_MARKETING_CSV',
    repo_root / 'data/kaggle/raw/bank-marketing/bank-full.csv',
))
output_dir = Path(os.environ.get(
    'BANK_MARKETING_PROCESSED_DIR',
    repo_root / 'data/kaggle/processed/bank-marketing',
))
seed = int(os.environ.get('BANK_MARKETING_SEED', '20260629'))
prepared = prepare_bank_marketing(csv_path, seed=seed)


## Dimensões e target

In [ ]:
print('Dimensões')
print({'linhas': len(prepared.features), 'features': len(prepared.metadata['feature_columns'])})
print('Distribuição do target')
target_distribution = Counter(prepared.target)
print({'no': target_distribution[0], 'yes': target_distribution[1], 'taxa_yes': prepared.metadata['target_mean']})


## Qualidade básica das features preparadas

In [ ]:
print('Qualidade básica')
quality = {}
for column in prepared.metadata['feature_columns']:
    values = [row[column] for row in prepared.features]
    quality[column] = {
        'ausentes': sum(value is None or value == '' for value in values),
        'unknown': sum(value == 'unknown' for value in values),
        'valores_distintos': len(set(values)),
    }
print(json.dumps(quality, ensure_ascii=False, indent=2))


## Decisões de tratamento e linhagem

In [ ]:
print('Decisões de tratamento')
treatment = {
    'features_mantidas': prepared.metadata['feature_columns'],
    'colunas_excluidas': prepared.metadata['excluded_columns'],
    'vazamento_temporal_removido': prepared.metadata['temporal_leakage_columns'],
    'pdays_negativo': 'convertido para ausente',
    'campaign': 'limitado ao intervalo de 0 a 10 contatos',
    'ordem_experimental': f'embaralhada deterministicamente com seed {seed}',
}
print(json.dumps(treatment, ensure_ascii=False, indent=2))
print('Linhagem')
print(json.dumps(prepared.metadata, ensure_ascii=False, indent=2))


## Exportação para o experimento offline

In [ ]:
output_dir.mkdir(parents=True, exist_ok=True)
features_path = output_dir / 'features.jsonl'
target_path = output_dir / 'target.csv'
metadata_path = output_dir / 'metadata.json'
features_path.write_text(
    ''.join(json.dumps(row, ensure_ascii=False) + '\n' for row in prepared.features),
    encoding='utf-8',
)
target_path.write_text(
    'y\n' + ''.join(f'{value}\n' for value in prepared.target),
    encoding='utf-8',
)
metadata_path.write_text(
    json.dumps(prepared.metadata, ensure_ascii=False, indent=2) + '\n',
    encoding='utf-8',
)
print({'features': str(features_path), 'target': str(target_path), 'metadata': str(metadata_path)})


## Limitação

Os artefatos preparados continuam sendo proxies públicos de marketing. Eles não medem causalidade, risco de crédito, elegibilidade real nem aderência a jornadas reais de Empréstimos com Garantia.